In [1]:
import pandas as pd
from pathlib import Path

In [11]:

rel_path = Path("/home/daniel.lien/dev/homework/data/")
dataset_csv = Path("new_copd_data.csv") # Start with Cleaned COPD dataset 


In [12]:
df = pd.read_csv(rel_path / dataset_csv)
print(df.shape)

(91220, 12)


In [13]:
# Create Question Mapping 

# Create a dictionary where QuestionID maps to a list of unique Questions
question_mapping = df.groupby("QuestionID")["Question"].unique().apply(list).to_dict()

# Check if each QuestionID has only one unique Question
question_id_counts = df.groupby("QuestionID")["Question"].nunique()

# Identify QuestionIDs with multiple unique questions
multiple_questions = question_id_counts[question_id_counts > 1]

# Print results
print("Question Mapping Dictionary:")
print(question_mapping)

if not multiple_questions.empty:
    print("\n🚨 Warning: Some QuestionIDs have multiple unique Questions:")
    print(multiple_questions)
else:
    print("\n✅ Each QuestionID maps to one unique Question.")

Question Mapping Dictionary:
{'COPD1_1': ['Mortality with chronic obstructive pulmonary disease as underlying cause among adults aged >= 45 years'], 'COPD1_2': ['Mortality with chronic obstructive pulmonary disease as underlying or contributing cause among adults aged >= 45 years'], 'COPD2_0': ['Prevalence of chronic obstructive pulmonary disease among adults >= 18'], 'COPD2_0_1': ['Prevalence of chronic obstructive pulmonary disease among adults >= 45 years'], 'COPD3_0': ['Prevalence of current smoking among adults >= 18 with diagnosed chronic obstructive pulmonary disease'], 'COPD3_0_1': ['Prevalence of current smoking among adults >= 45 years with diagnosed chronic obstructive pulmonary disease'], 'COPD4_0': ['Prevalence of activity limitation among adults >= 18 with diagnosed chronic obstructive pulmonary disease'], 'COPD4_0_1': ['Prevalence of activity limitation among adults >= 45 years with diagnosed chronic obstructive pulmonary disease'], 'COPD5_1': ['Hospitalization for chron

In [14]:
# Question Mapping Dictionary:
{
'COPD1_1': ['Mortality with chronic obstructive pulmonary disease as underlying cause among adults aged >= 45 years'],
'COPD1_2': ['Mortality with chronic obstructive pulmonary disease as underlying or contributing cause among adults aged >= 45 years'],
'COPD2_0': ['Prevalence of chronic obstructive pulmonary disease among adults >= 18'],
'COPD2_0_1': ['Prevalence of chronic obstructive pulmonary disease among adults >= 45 years'],
'COPD3_0': ['Prevalence of current smoking among adults >= 18 with diagnosed chronic obstructive pulmonary disease'],
'COPD3_0_1': ['Prevalence of current smoking among adults >= 45 years with diagnosed chronic obstructive pulmonary disease'],
'COPD4_0': ['Prevalence of activity limitation among adults >= 18 with diagnosed chronic obstructive pulmonary disease'],
'COPD4_0_1': ['Prevalence of activity limitation among adults >= 45 years with diagnosed chronic obstructive pulmonary disease'],
'COPD5_1': ['Hospitalization for chronic obstructive pulmonary disease as first-listed diagnosis'],
'COPD5_2': ['Hospitalization for chronic obstructive pulmonary disease as any diagnosis'],
'COPD5_3': ['Hospitalization for chronic obstructive pulmonary disease as first-listed diagnosis among Medicare-eligible persons aged >= 65 years'],
'COPD5_4': ['Hospitalization for chronic obstructive pulmonary disease as any diagnosis among Medicare-eligible persons aged >= 65 years'],
'COPD6_1': ['Emergency department visit rate for chronic obstructive pulmonary disease as first-listed diagnosis'],
'COPD6_2': ['Emergency department visit rate for chronic obstructive pulmonary disease as any diagnosis'],
'COPD7_0': ['Influenza vaccination among noninstitutionalized adults aged >= 45 years with chronic obstructive pulmonary disease'],
'COPD8_0': ['Pneumococcal vaccination among noninstitutionalized adults aged >= 45 years with chronic obstructive pulmonary disease']
}



{'COPD1_1': ['Mortality with chronic obstructive pulmonary disease as underlying cause among adults aged >= 45 years'],
 'COPD1_2': ['Mortality with chronic obstructive pulmonary disease as underlying or contributing cause among adults aged >= 45 years'],
 'COPD2_0': ['Prevalence of chronic obstructive pulmonary disease among adults >= 18'],
 'COPD2_0_1': ['Prevalence of chronic obstructive pulmonary disease among adults >= 45 years'],
 'COPD3_0': ['Prevalence of current smoking among adults >= 18 with diagnosed chronic obstructive pulmonary disease'],
 'COPD3_0_1': ['Prevalence of current smoking among adults >= 45 years with diagnosed chronic obstructive pulmonary disease'],
 'COPD4_0': ['Prevalence of activity limitation among adults >= 18 with diagnosed chronic obstructive pulmonary disease'],
 'COPD4_0_1': ['Prevalence of activity limitation among adults >= 45 years with diagnosed chronic obstructive pulmonary disease'],
 'COPD5_1': ['Hospitalization for chronic obstructive pulmon

In [15]:
# test with COPD1_1
filtered_df = df[(df["QuestionID"] == "COPD1_1")]
filtered_df.to_csv(rel_path / "COPD1_1.csv", index=False)
print(filtered_df.shape)

(9204, 12)


In [18]:
pivot_df = df.pivot_table(
    index=["YearStart", "LocationDesc", "Topic", "Question",
            "DataValueType", "DataValueUnit"],
    columns="StratificationCategory1",
    values="DataValue",
    aggfunc="first"  # Use 'first' to keep the first value encountered (or other aggregation method if needed)
)
pivot_df.to_csv(rel_path / "pivot.csv")

In [19]:
# Handle the 'Rate' DataValueType by dividing it by the 'DataValueUnit'
df["DataValue"] = df.apply(
    lambda row: row["DataValue"] / row["DataValueUnit"]
    if row["DataValueType"] == "Rate" else row["DataValue"],
    axis=1
)

# Re-pivot the table after applying transformations
pivot_df = df.pivot_table(
    index=["YearStart", "LocationDesc", "Topic", "Question"],
    columns="StratificationCategory1",
    values="DataValue",
    aggfunc="first"  # or another aggregation method
)

# Now, to group by year and state to calculate averages
aggregated_df = pivot_df.groupby(["YearStart", "LocationDesc"]).mean().reset_index()

# Save the aggregated data to a CSV
aggregated_df.to_csv(rel_path / "aggregated_copd_data.csv", index=False)

print("Aggregated data saved as 'aggregated_copd_data.csv'.")

Aggregated data saved as 'aggregated_copd_data.csv'.
